# Loading

In [1]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/jobs.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

5


In [3]:
type(docs[0])

langchain_core.documents.base.Document

In [2]:
docs[0].page_content
docs[0].metadata

{'producer': 'Prince 16.1 (www.princexml.com)',
 'creator': 'PyPDF',
 'creationdate': '',
 'title': 'jobs',
 'source': 'data/jobs.pdf',
 'total_pages': 5,
 'page': 0,
 'page_label': '1'}

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
def load_pdf(pdf_path: str) -> list[Document]:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

In [6]:
documents=load_pdf(file_path)

In [5]:
len(documents[0].page_content)


NameError: name 'documents' is not defined

# Chunking

In [7]:
# token split
from langchain_text_splitters import TokenTextSplitter
token_splitter = TokenTextSplitter(chunk_size=50, chunk_overlap=20)
split_docs = token_splitter.split_documents(documents)
len(split_docs)
split_docs[0].page_content



'Danh sách JD của RikkeiSoft\n• 1. Frontend Developer (React/Next.js)\n◦ Mô tả công việc:\n�'

In [ ]:
# markdown split

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=["#"])
split_docs = markdown_splitter.split_text(documents[0].page_content)

In [8]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
split_docs = text_splitter.split_documents(documents)
print(len(split_docs))
print("Content: ", split_docs[0].page_content)
print("Metadata: ", split_docs[0].metadata)



38
Content:  Danh sách JD của RikkeiSoft
• 1. Frontend Developer (React/Next.js)
◦ Mô tả công việc:
▪ Phát triển giao diện web hiện đại, tối ưu trải nghiệm người dùng.
▪ Xây dựng các component tái sử dụng bằng React/Next.js.
▪ Tối ưu hiệu năng client-side, SEO và tốc độ tải trang.
Metadata:  {'producer': 'Prince 16.1 (www.princexml.com)', 'creator': 'PyPDF', 'creationdate': '', 'title': 'jobs', 'source': 'data/jobs.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
def load_and_split(path: str):
    loader = PyPDFLoader(path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    split_docs = splitter.split_documents(documents)
    return split_docs

# Embeddings

In [10]:
from openai import OpenAI
from config import settings
client = OpenAI(api_key=settings.LLM_API_KEY, base_url=settings.LLM_BASE_URL)
def embed_query(query: str):
    response = client.embeddings.create(input=query, model=settings.LLM_EMBEDDING_MODEL)
    return response.data[0].embedding


In [5]:
embed_query("Hello, world!")


[-0.020920176,
 0.009755219,
 0.004780903,
 -0.059421252,
 0.0050828396,
 0.000864511,
 -0.0045847003,
 0.0050770855,
 0.019296365,
 0.01731793,
 -0.008134706,
 -0.00904072,
 -0.004410189,
 0.030996358,
 0.10537974,
 0.010174653,
 0.023028461,
 -0.028398033,
 0.0023683798,
 -0.006660489,
 0.0028257493,
 -0.001545771,
 0.033225145,
 -0.008438869,
 0.0032804706,
 0.007675922,
 0.036734413,
 -0.016130663,
 0.025337653,
 0.021699391,
 -0.00081538834,
 0.015893541,
 -0.045027964,
 0.007590516,
 9.4052295e-05,
 0.03321805,
 -0.0026741568,
 -0.014441575,
 -0.004091568,
 0.008324071,
 -0.012123116,
 -0.0029273992,
 0.007087799,
 -0.017040461,
 -0.007887804,
 0.004777341,
 -0.021534339,
 -0.026621556,
 0.00946892,
 0.006584679,
 -0.014829595,
 0.010655077,
 -0.021971917,
 -0.16638611,
 -0.0060834247,
 0.010029496,
 0.0013234994,
 0.017470278,
 0.015293622,
 0.0024907223,
 -0.0077383164,
 0.017951336,
 -0.018513493,
 -0.025343783,
 -0.010616714,
 -0.0028965871,
 -0.006425442,
 0.014603657,
 -0.0

In [6]:
vector_1 = embed_query("Rikkesoft là gì?")
vector_2=embed_query("Rikkeisoft là công ty công nghệ!")
# Cosine similarity



In [7]:
vector_3 = embed_query("Hôm nay trời mưa")

In [11]:
import numpy as np

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


In [20]:
cosine_similarity(vector_1, vector_2)

np.float64(0.8053668919383327)

In [21]:
cosine_similarity(vector_1, vector_3)

np.float64(0.547987148239403)

# Vector Database

In [22]:
from typing import List, Optional

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from config import settings
from langchain_core.documents import Document
embeddings = GoogleGenerativeAIEmbeddings(
    api_key=settings.LLM_API_KEY,
    model=settings.LLM_EMBEDDING_MODEL,
)
def create_or_get_vector_store(
    collection_name: str = "documents",
    persist_directory: str = "lecture_2/vector_store",
    delete_existing: bool = False,
):
    if delete_existing:
        Chroma(
            persist_directory=persist_directory,
            collection_name=collection_name,
            embedding_function=embeddings,
        ).delete_collection(collection_name)
    return Chroma(
        persist_directory=persist_directory,
        collection_name=collection_name,
        embedding_function=embeddings,
    )


In [23]:
def ingest_documents(
    collection_name: str = "documents",
    persist_directory: str = "lecture_2/vector_store",
    documents: Optional[List[Document]] = None,
):
    vector_store = create_or_get_vector_store(collection_name, persist_directory)
    vector_store.add_documents(documents)
    return vector_store

In [25]:
documents = load_and_split("data/jobs.pdf")
print(f"Ingesting {len(documents)} documents from data/jobs.pdf")
ingest_documents(documents=documents)
print(f"Ingested {len(documents)} documents from data/jobs.pdf")

Ingesting 21 documents from data/jobs.pdf
Ingested 21 documents from data/jobs.pdf


# Retriever

In [26]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from typing import List
def get_retriever(query: str)-> List[Document]:
    vectorstore = create_or_get_vector_store()
    return vectorstore.similarity_search(query, k=3)

In [27]:
retriever = get_retriever("QA Engineer (Manual/Automation")
retriever[0].page_content

'▪ Biết sử dụng các công cụ CI/CD (GitHub Actions/GitLab CI/Jenkins...).\n▪ Hiểu về networking cơ bản, bảo mật hệ thống, Linux.\n▪ Kỹ năng scripting (Bash/Python) tốt.\n• 4. QA Engineer (Manual/Automation)\n◦ Mô tả công việc:\n▪ Xây dựng test plan, test case cho các tính năng mới.\n▪ Thực hiện test manual (functional, regression, UI/UX…).\n▪ Viết và duy trì test automation (API/UI) khi cần thiết.\n▪ Phối hợp với team Dev/PM để tái hiện và theo dõi bug.'

# RAG PIPELINE

In [30]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from typing import List, Tuple
from pydantic import BaseModel, Field
from config import settings
from src.retriever import get_retriever
from prompt.prompt import RAG_SYSTEM_PROMPT
# Output structure definition
class Source(BaseModel):
    """Source information"""
    document: int = Field(description="Document number")
    source: str = Field(description="Source file path or link")
    page: int = Field(description="Page number in the source document", default=None)
class RAGResponse(BaseModel):
    """Structured output for RAG responses"""
    answer: str = Field(description="The answer to the question based on context documents")
    sources: List[Source] = Field(description="List of sources with document number, source path, and page number")


def _format_documents(docs: List[Document]) -> str:
    """Format retrieved documents into JSON string format"""
    documents_list = []
    for i, doc in enumerate(docs, 1):
        doc_dict = {
            "document": i,
            "source": doc.metadata.get("source", "Unknown"),
            "page": doc.metadata.get("page", None),
            "content": doc.page_content.strip()
        }
        documents_list.append(doc_dict)
    
    return json.dumps(documents_list, ensure_ascii=False, indent=2)


# Create output parser
output_parser = PydanticOutputParser(pydantic_object=RAGResponse)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "{question}")
])


def chat_with_rag(query: str) -> Tuple[RAGResponse, List[Document]]:
    llm = ChatOpenAI(
        api_key=settings.LLM_API_KEY,
        model=settings.LLM_CHAT_MODEL,
        base_url=settings.LLM_BASE_URL,
        temperature=0.1
    )
    
    docs = get_retriever(query)
    context = _format_documents(docs)
    
    # Chain with structured output parser
    chain = RAG_PROMPT | llm | output_parser
    
    result = chain.invoke({"question": query, "context": context})
    return result, docs


In [ ]:
chat_with_rag("Trong công ty có những vị trí gì")

(RAGResponse(answer='Dựa trên các tài liệu được cung cấp, công ty có các vị trí sau:\n1. Data Scientist / Machine Learning Engineer (vị trí số 8).\n2. Product Manager (vị trí số 5).\n3. Technical Lead / Software Architect (vị trí số 10).\n4. Một vị trí liên quan đến kiểm thử phần mềm (QA/Tester) với các yêu cầu như kinh nghiệm kiểm thử, viết test case, test report và sử dụng tool automation.\nNgoài ra, tài liệu còn nhắc đến sự phối hợp với team Dev và PM.', sources=[Source(document=1, source='data/jobs.pdf', page=2), Source(document=2, source='data/jobs.pdf', page=1), Source(document=3, source='data/jobs.pdf', page=3)]),
 [Document(id='66dcef40-ff18-4a84-9c71-4cde55234f4e', metadata={'page': 2, 'total_pages': 5, 'title': 'jobs', 'creator': 'PyPDF', 'source': 'data/jobs.pdf', 'producer': 'Prince 16.1 (www.princexml.com)', 'page_label': '3', 'creationdate': ''}, page_content='• 8. Data Scientist / Machine Learning Engineer\n◦ Mô tả công việc:'),
  Document(id='4959f908-224b-40fb-90e5-d1d